<table  align="left" width="100%"> <tr>
        <td  style="background-color:#ffffff;"><a href="https://qworld.net" target="_blank"><img src="../images/qworld/qworld.jpg" width="35%" align="left"></a></td>
        <td  align="right" style="background-color:#ffffff;vertical-align:bottom;horizontal-align:right">
            prepared by Özlem Salehi Köken
        </td>        
</tr></table>

# Variational quantum algorithms

Variational Quantum Algorithms (VQAs) are a class of hybrid quantum-classical algorithms designed to harness the power of quantum computing while addressing the challenges posed by current quantum hardware.

These algorithms are particularly promising for near-term quantum devices, known as Noisy Intermediate-Scale Quantum (NISQ) devices, which are limited by noise, short coherence times, and relatively small numbers of qubits.

At their core, VQAs combine quantum and classical computing resources in an iterative process to solve complex problems.

In this module, we will explore the principles behind VQAs, the role of quantum and classical components, and how these algorithms are implemented on current quantum hardware.

## Parametrized Quantum Circuits (PQCs)

Parametrized quantum circuits consist of quantum gates that have tunable parameters, allowing the quantum state to be manipulated during optimization.

In the context of VQAs, PQCs are crucial because they allow the quantum system to be "tuned" in a way that makes the system more suitable for the problem at hand.

Quantum gates with parameters are those whose parameters are typically represented as angles (e.g., $R_x(\theta), R_y(\theta),\dots$
).

The goal is to optimize these parameters to minimize or maximize a specific cost function, which is often tied to the problem being solved (such as finding the ground state energy in quantum chemistry or solving optimization problems).

<img src="../images/vqa1.png" width="90%" align="center">

For example, in Quantum Chemistry applications (such as the Variational Quantum Eigensolver), PQCs are used to model the quantum states of molecules. These circuits prepare trial states that can then be used to estimate the ground state energy of a molecular system.

In Optimization Problems (such as with the Quantum Approximate Optimization Algorithm), PQCs represent potential solutions to optimization problems. Classical optimization techniques adjust the parameters of the quantum circuit iteratively based on the outcomes of quantum measurements, with the goal of finding the optimal solution.

For Machine Learning applications, PQCs can represent the parameterized models in quantum neural networks. These models are trained to perform tasks such as classification, regression, or clustering.

### Creating parametrized circuits in Qiskit

In Qiskit, one can create circuits with parameters, without providing explicit values to the parameters of the gates. This can be achieved by the `Parameter` object from Qiskit.

In [ ]:
from qiskit.circuit import Parameter

phi = Parameter(name = 'phi')

The created parameter can be used for instance with rotaton gates as follows.

In [ ]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(1)
qc.rz(phi, 0)

qc.draw(output='mpl')

### Task 1

Create a PQC with 2 qubits in Qiskit. Implement a $R_x$ rotation on qubit 0 with parameter $\theta_0$ and $R_y$ rotation on qubit 1 with parameter $\theta_1$. Add a CNOT gate where control is qubit 0 and target is qubit 1.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

# Define parameters for the PQC
theta_0 = Parameter('theta_0')
theta_1 = Parameter('theta_1')

### Your code here

# Visualize the quantum circuit
qc.draw(output="mpl")


[click for our solution](01_variational_quantum_algorithms_solutions.ipynb#task1)

Before simulating a circuit, the parameters should be "bind" to some value. This can be achieved using the function `assign_parameters`. The function can be called either by the keyword `inplace=True` or the returned circuit should be saved.

In [ ]:
from qiskit import  QuantumCircuit
from qiskit.circuit import Parameter

# Define parameters for the PQC
phi = Parameter('phi')

qc = QuantumCircuit(1)
qc.rz(phi, 0)
qc.assign_parameters({phi: 0.5}, inplace=True)

qc.draw(output='mpl')

### Taks 2

 Use the `assign_parameters` function to assign $\theta_0 = \pi$ and $\theta_1 =1$.

In [ ]:
from math import pi
from qiskit import  QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter

# Define parameters for the PQC
theta_0 = Parameter('theta_0')
theta_1 = Parameter('theta_1')

# Create a Quantum Circuit with 2 qubits
qc = QuantumCircuit(2)

# Apply parameterized gates
qc.rx(theta_0, 0)  # Apply RX gate with parameter theta_0 on qubit 0
qc.ry(theta_1, 1)  # Apply RY gate with parameter theta_1 on qubit 1
qc.cx(0, 1)        # Apply CNOT gate between qubits 0 and 1

### Your code here

# Visualize the quantum circuit
qc.draw(output="mpl")

[click for our solution](01_variational_quantum_algorithms_solutions.ipynb#task2)

## Ansatz

As mentioned above, the parametrized quantum circuits in VQAs often encode potential solutions to problems. Such a trial state, which is often referred to as an *ansatz* can be seen as a guess for the solution. Construction of an ansatz may be problem inspired, or totally problem-independent, which we will address later on.

One important aspect when choosing an ansatz is expressibility, which can be explained as the possible states that can be obtained by setting different parameters to the ansatz.

For instance, the set of states that are spanned applying $R_X(\theta)$ on a quantum circuit with a single qubit are visualized below on the Bloch sphere.

<img src="../images/bloch_ansatz.png" width="90%" align="center">

### Example

<img src="../images/ansatz_example.png" align="center">

For the 2-qubit circuit given above, let us express the set of states that can be expressed as a parameter of $\theta_0$ and $\theta_1$. Note that $R_x(\theta)$ and $R_y(\theta)$ gates are represented by the following matrices:

$$
R_x(\theta)  = \begin{pmatrix}
\cos\left(\frac{\theta}{2}\right) & -i \sin\left(\frac{\theta}{2}\right) \\
-i \sin\left(\frac{\theta}{2}\right) & \cos\left(\frac{\theta}{2}\right)
\end{pmatrix},
~~~~
R_y(\theta)  = \begin{pmatrix}
\cos\left(\frac{\theta}{2}\right) & - \sin\left(\frac{\theta}{2}\right) \\
 \sin\left(\frac{\theta}{2}\right) & \cos\left(\frac{\theta}{2}\right)
\end{pmatrix}.
$$

The initial state is given by
$\ket{\psi_0} = \ket{00}$. After applying $R_x(\theta_0)$ on qubit 0 and $R_x(\theta_1)$ on qubit 1, we get the following:

\begin{align*}
\ket{\Psi_1} &= \left (\cos\left(\frac{\theta_1}{2}\right)\ket{0} + \sin\left(\frac{\theta_1}{2}\right) \ket{1} \right ) \otimes \left (\cos\left(\frac{\theta_0}{2}\right)\ket{0} -i \sin\left(\frac{\theta_0}{2}\right) \ket{1} \right ) \\
&= \cos\left( \frac{\theta_1}{2} \right) \cos\left( \frac{\theta_0}{2} \right) \ket{00} - i \cos\left( \frac{\theta_1}{2} \right) \sin\left( \frac{\theta_0}{2} \right) \ket{01} + \sin\left( \frac{\theta_1}{2} \right) \cos\left( \frac{\theta_0}{2} \right) \ket{10} - i \sin\left( \frac{\theta_1}{2} \right) \sin\left( \frac{\theta_0}{2} \right) \ket{11}
\end{align*}

Next, let us apply the CNOT gate.
$$
\ket{\psi_2} = \cos\left( \frac{\theta_1}{2} \right) \cos\left( \frac{\theta_0}{2} \right) \ket{00}  - i \sin\left( \frac{\theta_1}{2} \right) \sin\left( \frac{\theta_0}{2} \right) \ket{01} + \sin\left( \frac{\theta_1}{2} \right) \cos\left( \frac{\theta_0}{2} \right) \ket{10} - i \cos\left( \frac{\theta_1}{2} \right) \sin\left( \frac{\theta_0}{2} \right) \ket{11}
$$

Setting both parameters to $2\pi$, for instance, yields the state $\ket{00}$.



### Task 3
The quantum state of the circuit in Task 1 can be expressed as

$\cos\left( \frac{\theta_1}{2} \right) \cos\left( \frac{\theta_0}{2} \right) \ket{00}  - i \sin\left( \frac{\theta_1}{2} \right) \sin\left( \frac{\theta_0}{2} \right) \ket{01} + \sin\left( \frac{\theta_1}{2} \right) \cos\left( \frac{\theta_0}{2} \right) \ket{10} - i \cos\left( \frac{\theta_1}{2} \right) \sin\left( \frac{\theta_0}{2} \right) \ket{11}.$

Set both parameters to $2\pi$ and verify that the output is $\ket{00}$. Check also mathematically that this is the case.


In [ ]:
from math import pi
from qiskit import  QuantumCircuit
from qiskit.circuit import Parameter

# Define parameters for the PQC
theta_0 = Parameter('theta_0')
theta_1 = Parameter('theta_1')

# Create a Quantum Circuit with 2 qubits
qc = QuantumCircuit(2)

# Apply parameterized gates
qc.rx(theta_0, 0)  # Apply RX gate with parameter theta_0 on qubit 0
qc.ry(theta_1, 1)  # Apply RY gate with parameter theta_1 on qubit 1
qc.cx(0, 1)        # Apply CNOT gate between qubits 0 and 1

### Your code here

job = AerSimulator().run(qc,shots=1024)
counts = job.result().get_counts(qc)
print(counts)



[click for our solution](01_variational_quantum_algorithms_solutions.ipynb#task3)

### Task 4

You are given a function that takes as parameter a parameter list of length 2. Assign the parameters to the `qc` and execute with `shots=1024`. The function should return the probability of **not** obtaining state $\ket{00}$ based on the measurement outcomes.

In [ ]:
from qiskit import  QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter

def func(params):
    # Define parameters for the PQC
    theta_0 = Parameter('theta_0')
    theta_1 = Parameter('theta_1')

    # Create a Quantum Circuit with 2 qubits
    qc = QuantumCircuit(2)

    # Apply parameterized gates
    qc.rx(theta_0, 0)  # Apply RX gate with parameter theta_0 on qubit 0
    qc.ry(theta_1, 1)  # Apply RY gate with parameter theta_1 on qubit 1
    qc.cx(0, 1)        # Apply CNOT gate between qubits 0 and 1

    ### Your code here

    return 1- prob

[click for our solution](01_variational_quantum_algorithms_solutions.ipynb#task4)

## Optimization in VQAs

In VQAs, optimization of the parameters is essential because the quantum circuit itself doesn’t "know" the correct parameters, it only generates a quantum state based on its parameters, and it’s the classical optimizer’s job to refine those parameters to minimize or maximize a given cost function.




<img src="../images/optimization.png" align="center">


### How Classical Optimizers Guide Quantum Circuits
In VQAs, a quantum circuit is initialized with some parameters, and the output of the circuit is measured. These measurement results are used to compute a cost function, which tells us how good the current parameters are. A classical optimizer then adjusts the parameters to improve the result.






The cost function represents what we are trying to minimize or maximize. For example, in quantum chemistry, the cost function might be the energy of a molecule, and in optimization problems, it could represent a score that needs to be minimized. The quantum circuit is run multiple times with different parameters to evaluate how the cost function changes, helping the optimizer decide how to update the parameters.

The `scipy` library offers classical optimizers that adjust numerical parameters in order to minimize (or maximize) a cost function. SciPy provides several optimization methods through `scipy.optimize`, including:

 - COBYLA – a gradient-free optimizer well-suited for noisy cost functions.

- Nelder-Mead – a simplex-based method that does not require gradients.

- BFGS and L-BFGS-B – gradient-based methods that can converge faster when gradients are available.

- SLSQP – supports constraints on parameters.

In practice, the cost function is defined as a Python function that takes a vector of parameters as input and returns a scalar value. SciPy repeatedly calls this function, evaluates the result, and updates the parameters until convergence criteria are satisfied.

Here is an example usage of the `scipy.optimize`.

In [ ]:
from scipy.optimize import minimize

# We need to make an initial guess for the parameters that will be optimized
initial_guess = [3 , 1]

# Define the objective function to be minimized
def f(x): #x is the list of parameters to be optimized
    x1, x2 = x
    return (x1*x2-4)**2

#We provide the objective function and the initial guess to the optimization algorithm
result = minimize(f, initial_guess, method='Nelder-Mead') 


Let us inspect the result object.

In [ ]:
result

In [ ]:
result.x

In this example, we expect $x_1x_2=4$ should be satisfied so that $f(x)$ gets its minimum value, which is 0. You can change the initial paraneters and check how they effect the result.

### Task 5

Consider the following function named `himmelblau`. Use `scipy.minimize` to find minimium of this function. Try different methods for optimization and the following initial values:
- (-2.8, 3)
- (3.1, 2.1)
- (-3.7, -3.2)
- (3.5, -1.8)
- random one


<img src="../images/Himmelblau_function.png" align="center">

In [ ]:
def himmelblau(variables):
    x, y = variables
    return (x**2 + y - 11)**2 + (x + y**2 - 7)**2

In [ ]:
from scipy.optimize import minimize
import numpy as np

# Set params
initial_params = 
print("Initial parameters:", initial_params )

for method in ["COBYLA", "Nelder-Mead", "L-BFGS-B", "BFGS", "SLSQP"]:
    #Your code here

[click for our solution](01_variational_quantum_algorithms_solutions.ipynb#task5)

The function given above has 4 minima at

 (3, 2)

(-2.805118, 3.131312)

(-3.779310, -3.283186)

(3.584428, -1.848126)

all 4 giving 0 for the function value.

Now we will use the minimize function to tune the parameters of a parameterized quantum circuits.

### Task 6

In this task, you will optimize the parameters of the quantum circuit given in Task 1, so that the quantum circuit yields state $|00\rangle$.

You will use scipy library to get a classical optimizer. In this task, we select the COBYLA method. 

Assign the initial parameters randomly and use the function defined in Task 4 to define a cost function to be used by the optimizer.


In [ ]:
from scipy.optimize import minimize

initial_guess = ### Your code here

cost_function = ### Your code here

result = minimize(cost_function, initial_guess, method='COBYLA')
print("Optimization results:", result)

[click for our solution](01_variational_quantum_algorithms_solutions.ipynb#task6)

### Task 7

Use the parameters found in Task 6 and assign them to the circuit in Task 1. Verify whether state $\ket{00}$ is observed with large probability.

In [ ]:
from math import pi
from qiskit import  QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.circuit import Parameter



### Your code here

job = AerSimulator().run(qc,shots=1024)
counts = job.result().get_counts(qc)
print(counts)

[click for our solution](01_variational_quantum_algorithms_solutions.ipynb#task5)